In [ ]:
import os
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as playwrightTimeout
import time
import asyncio

In [ ]:
%pip install beautifulsoup4
%pip install playwright
!playwright install 

In [ ]:
SEASONS = list(range(2020,2026))
DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR,"standings")
SCORES_DIR = os.path.join(DATA_DIR,"scores")

In [ ]:
#funtion to get html of a given page 
async def html_getter(url, selector, sleep = 5, retries = 4 ): 
    html = None
    for i in range(1,retries):
        time.sleep(sleep*i) #slow down scraping to not get banned
        try:
            async with async_playwright() as p:
                browser = await p.firefox.launch() #wait for browser
                tab = await browser.new_page()
                await tab.goto(url)
                print(await tab.title())
                html = await tab.inner_html(selector)
        except playwrightTimeout:
            print(f"timeout error on {url}")
            continue
        else:
            break
    return html


In [ ]:
async def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = await html_getter(url,"#content .filter")  
    soup =  BeautifulSoup(html)
    links = soup.find_all("a")
    href = [l["href"] for l in links]
    standing_pages = [f"https://www.basketball-reference.com/{l}" for l in href]
    #now have all links for each month of a season

    for url in standing_pages:
        save_path = os.path.join(STANDINGS_DIR,url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        html = await html_getter(url, "#all_schedule")
        with open(save_path, "w+") as f:
            f.write(html)


In [ ]:
for season in SEASONS:
    await scrape_season(season)

In [ ]:
standings_files = os.listdir(STANDINGS_DIR)

In [ ]:
async def scrape_box(standings_file):
    with open(standings_file, 'r') as f:
        html = f.read()
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    href = [l.get("href") for l in links]
    box_scores = [l for l in href if l and "boxscore" in l and ".html" in l]
    box_scores =  [f"https://www.basketball-reference.com/{l}" for l in box_scores]

    for url in box_scores:
        save_path = os.path.join(SCORES_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue

        html = await html_getter(url, "#content")
        if not html:
            continue
        with open(save_path,"w+") as f:
            f.write(html)
       

In [ ]:
standings_files =  [sf for sf in standings_files if ".html" in sf]

In [ ]:
for f in standings_files:
    filepath = os.path.join(STANDINGS_DIR, f)

    await scrape_box(filepath)